# Dwarf Example 15: Four-Curve Rotation Plot

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

Reproduce the EPS four-curve diagnostic for a dwarf:
Vobs, V_adj, V_Kep, and a proxy baryonic curve.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
with open('dwarf_irregular_corpus_v1.json') as f:
    corpus = json.load(f)
omega_ready=[g for g in corpus['galaxies']
             if g.get('omega_ready') and g.get('data') and len(g['data'])>=4]
g=sorted(omega_ready,key=lambda x:len(x['data']),reverse=True)[0]
d=g['data']; R=np.array([p['Rad'] for p in d]); V=np.array([p.get('Vrot', 0) for p in d])
errV=np.array([p.get('errV',0) for p in d])
R1,V1=R[0],V[0]; R2,V2=R[-1],V[-1]
omega=V2/R2 - (V1/R1)*(R1/R2)**1.5  # Eq.6 corrected 2026-07-12: operator-precedence fix
V_adj=V-R*omega; V_kep=np.sqrt(V2**2*R2/R)
fig,ax=plt.subplots(figsize=(8,5))
ax.errorbar(R,V,yerr=errV,fmt='o',color='#2166ac',capsize=3,ms=5,
            label=r'$V_{\rm obs}$',zorder=5)
ax.plot(R,V_adj,'^-',color='#2ca02c',lw=1.8,
        label=fr'$V_{{\rm adj}}$ ($\omega={omega:.2f}$ rad/Gyr)')
ax.plot(R,V_kep,'--',color='#ff7f0e',lw=1.2,label='Keplerian')
ax.set_xlabel('Radius (kpc)',fontsize=12); ax.set_ylabel('km/s',fontsize=12)
ax.set_title(f'{g["galaxy"]} — EPS Four-Curve Plot\nDwarf Corpus v1.0',fontsize=10)
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('dw15_four_curve.png',dpi=150,bbox_inches='tight'); plt.show()
print(f"Galaxy: {g['galaxy']}, omega={omega:.3f} rad/Gyr")